# The Gauss–Seidel Method
The Gauss–Seidel method updates unknowns in sequence, using each new value as soon as it is available. This can reduce the number of iterations compared with Jacobi, but the benefit depends on the matrix and the ordering of the unknowns.

> __Learning Objectives:__
>
> By the end of this algorithm notebook, you should be able to:
> - **Derive the update:** Use the diagonal and lower triangular parts of the system matrix to construct a residual correction.
> - **Follow a sequential sweep:** Identify which components use new values and which still use values from the previous iteration.
> - **Assess convergence and stopping:** Identify the iteration matrix and distinguish meeting the residual tolerance from reaching a correction limit.

This notebook specializes the [general iterative method](CHEME-5800-L6c-Lecture-GeneralIterativeMethod-Fall-2026.ipynb). The [companion example](CHEME-5800-L6c-Example-FunWithIterativeSolvers-Fall-2026.ipynb) compares its numerical results and computational cost with other solvers.

___

## Matrix splitting and sequential updates
Consider $\mathbf{A}\mathbf{x}=\mathbf{b}$, where $\mathbf{A}\in\mathbb{R}^{n\times n}$ is nonsingular, $\mathbf{x}\in\mathbb{R}^{n}$ contains the $n$ unknowns, and $\mathbf{b}\in\mathbb{R}^{n}$ is the right-hand side. Assume every diagonal entry $a_{ii}$ is nonzero. Let $\mathbf{D}$, $\mathbf{L}$, and $\mathbf{U}$ contain the diagonal, strictly lower triangular, and strictly upper triangular entries of $\mathbf{A}$. The Gauss–Seidel splitting is:

$$
\mathbf{A}
=\underbrace{(\mathbf{D}+\mathbf{L})}_{\mathbf{M}}
+\underbrace{\mathbf{U}}_{-\mathbf{N}},
\qquad \mathbf{N}=-\mathbf{U}.
$$

At iteration $k$, define the residual by $\mathbf{r}^{(k)}=\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)}$. We compute the correction $\mathbf{d}^{(k)}$ and next approximation by:

$$
\begin{aligned}
(\mathbf{D}+\mathbf{L})\mathbf{d}^{(k)}&=\mathbf{r}^{(k)},
&&\text{solve by forward substitution},\\
\mathbf{x}^{(k+1)}&=\mathbf{x}^{(k)}+\mathbf{d}^{(k)},
&&\text{apply the correction}.
\end{aligned}
$$

Because $\mathbf{D}+\mathbf{L}$ is lower triangular with nonzero diagonal entries, this solve does not require an explicit inverse. Substituting the residual and collecting terms gives the equivalent update:

$$
(\mathbf{D}+\mathbf{L})\mathbf{x}^{(k+1)}
=\mathbf{b}-\mathbf{U}\mathbf{x}^{(k)}.
$$

Reading row $i$ explains how the sequential sweep works:

$$
x_i^{(k+1)}
=\frac{1}{a_{ii}}\left(
 b_i-\sum_{j<i}a_{ij}x_j^{(k+1)}
    -\sum_{j>i}a_{ij}x_j^{(k)}
\right),\qquad i=1,\ldots,n.
$$

The first sum uses values already updated during this sweep; the second uses values from the previous iteration. Empty sums are zero. Process the rows in order so each new component is available when the next row needs it.

___

## Algorithm and convergence
__Initialize__: Given the system matrix $\mathbf{A}\in\mathbb{R}^{n\times n}$ and right-hand side $\mathbf{b}\in\mathbb{R}^{n}$, choose an initial guess $\mathbf{x}^{(0)}\in\mathbb{R}^{n}$, an absolute residual tolerance $\epsilon>0$, and a nonnegative integer correction limit $\texttt{maxiter}$. Set $\texttt{converged}\gets\texttt{false}$ and the correction counter $k\gets0$.

While not $\texttt{converged}$ __do__:

1. Calculate the residual vector $\mathbf{r}^{(k)}\gets\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)}$.
2. Check for convergence:
   - If $\|\mathbf{r}^{(k)}\|_2<\epsilon$, __then__: set $\texttt{converged}\gets\texttt{true}$ and return $\mathbf{x}^{(k)}$ with this status.
   - Otherwise, if $k\ge\texttt{maxiter}$, __then__: print a __warning__ that the residual tolerance was not met and return $\mathbf{x}^{(k)}$ with $\texttt{converged}=\texttt{false}$.
3. Calculate the update direction: solve $(\mathbf{D}+\mathbf{L})\mathbf{d}^{(k)}=\mathbf{r}^{(k)}$ by forward substitution.
4. Update the solution vector: $\mathbf{x}^{(k+1)}\gets\mathbf{x}^{(k)}+\mathbf{d}^{(k)}$.
5. Increment the correction counter: $k\gets k+1$.

Check the residual before the correction limit: an acceptable initial guess needs no update, and an approximation that first meets tolerance after the last allowed correction is still successful. Either return stops the algorithm immediately. A small residual means the equations are nearly satisfied; its relationship to solution error also depends on the conditioning of $\mathbf{A}$.

To assess convergence of the underlying iteration, write the update in stationary form:

$$
\begin{aligned}
\mathbf{x}^{(k+1)}
&=\underbrace{-(\mathbf{D}+\mathbf{L})^{-1}\mathbf{U}}_{\mathbf{G}_{GS}}
\mathbf{x}^{(k)}
+\underbrace{(\mathbf{D}+\mathbf{L})^{-1}\mathbf{b}}_{\mathbf{c}}.
\end{aligned}
$$

Here $\mathbf{G}_{GS}$ is the iteration matrix and $\mathbf{c}$ is the constant vector. The minus sign follows from $\mathbf{N}=-\mathbf{U}$. The iteration converges to the solution from every initial guess if and only if:

$$
\rho(\mathbf{G}_{GS})=\max_i|\lambda_i|<1,
$$

where $\lambda_i$ are its eigenvalues and $\rho$ is the spectral radius. Strict row diagonal dominance is one sufficient condition; the [lecture's convergence section](CHEME-5800-L6c-Lecture-GeneralIterativeMethod-Fall-2026.ipynb) explains how this differs from a necessary condition.

___

## Summary
We specialized the residual-correction method to a lower triangular solve.

> __Key Takeaways:__
>
> - **Triangular corrections:** We derived the Gauss–Seidel correction from the diagonal and lower triangular parts of the system matrix. Forward substitution computes this correction without forming an inverse.
> - **Sequential updates:** We showed why each row uses newly computed values for earlier unknowns. This distinguishes Gauss–Seidel from Jacobi and makes the ordering of the sweep relevant.
> - **Convergence and stopping:** We derived the signed iteration matrix and stated its convergence condition. Our pseudocode returns immediately when the residual meets tolerance or the correction limit is reached, reporting these outcomes separately.

Use the companion example to check the final residual before comparing solver timings.

___